# Hafta 3 — Kaggle CLIP Embedding Cikarma

**Amac:** 17.176 laundered goruntu icin dondurulmus CLIP ViT-L/14 embedding'i
cikarip `.npy` olarak indirmek. Sonrasindaki her sey (grid search, esik secimi,
kalibrasyon, E3/E5/E6) yerelde **dakikalar** icinde doner.

## Neden goruntuleri yuklemiyoruz

Laundered goruntuler ~5-6 GB. Bunun yerine Kaggle'da **yeniden uretiyoruz**:
`cardd-data` zaten Kaggle'da duruyor, uretilmis S/M katmanlari 585 MB'lik iki
zip. Laundering yerelde 90 saniye surmustu -- Kaggle'da da hizli.

Uretilen manifest'in yereldekiyle AYNI oldugu, **dondurulmus test setinin
sha256'si** ile dogrulanir (hucre 4). Tutmazsa embedding'ler hizalanmaz ve
durmamiz gerekir.

## Onkosullar (calistirmadan once)

1. **Accelerator:** GPU T4 x2 (tek T4 de yeter, bu is tek GPU kullanir)
2. **Input olarak eklenmis 2 dataset:**
   - `cardd-data` (W2'den, zaten ekli olmali)
   - **YENI:** `w2-uretim` — `w2_synthetic.zip` + `w2_manipulated.zip`
     dosyalarini iceren bir dataset. Bunlari bilgisayarindan yukle:
     *Add Input -> Upload -> Create Dataset*
3. Internet: **acik** (repo klonlama + CLIP agirliklari icin)


## 0. Ortam ve GPU

In [ ]:
import os, sys, platform, subprocess, torch
from pathlib import Path

print("Python :", platform.python_version())
print("torch  :", torch.__version__)
print("GPU sayisi:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB")

import shutil
print("\nDisk (/kaggle/working):", f"{shutil.disk_usage('/kaggle/working').free/1e9:.0f} GB bos")

WORK = Path("/kaggle/working/insurance-image-forensics")
INPUT = Path("/kaggle/input")
print("\nInput datasetleri:")
for d in sorted(INPUT.iterdir()):
    print("  ", d.name)

In [ ]:
!pip install -q transformers accelerate safetensors pyarrow 2>&1 | tail -3
import transformers, sklearn
print("transformers", transformers.__version__)
print("sklearn", sklearn.__version__)

## 1. Repo, veri ve uretim ciktilari

In [ ]:
REPO_URL = "https://github.com/Tunahan-46/insurance-image-forensics.git"

if not WORK.exists():
    !git clone -q $REPO_URL {str(WORK)}
else:
    print("Repo zaten var, guncelleniyor")
    !cd {str(WORK)} && git pull -q

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd())
!git log --oneline -1

In [ ]:
# CarDD'yi kodun bekledigi yere BAGLA (kopyalama yok)
#
# NEDEN find_dataset_root VAR
# ----------------------------
# Eski kod `INPUT.iterdir()` ile Input'un DOGRUDAN altindaki klasorleri
# ariyordu (her dataset = bir alt klasor varsayimi). Bu ortamda Input
# tek bir "datasets" klasoruyle SARILMIS -- gercek dataset'ler bunun
# icinde, muhtemelen bir veya iki seviye daha derinde. `INPUT.iterdir()`
# artik yalnizca ["datasets"] donduruyor, "cardd" hic eslemiyor ->
# StopIteration.
#
# Cozum: tum Input agacinda (herhangi bir derinlikte) rglob ile ara, adinda
# aranan kelime gecen EN SIG (en az alt-yol parcali) dizini sec. Sig
# olani secmek onemli: "cardd-data" kendisi de "cardd_coco",
# "CarDD_COCO", "CarDD-TR-Mask" gibi kendi ICINDEKI alt klasorler de
# hep "cardd" iceriyor -- ama onlar dataset KOKUNDEN daha derinde, o
# yuzden min(derinlik) her zaman gercek dataset kokunu secer.

def find_dataset_root(substr: str):
    cands = [p for p in INPUT.rglob("*") if p.is_dir() and substr in p.name.lower()]
    if not cands:
        return None
    return min(cands, key=lambda p: len(p.relative_to(INPUT).parts))


DATA = find_dataset_root("cardd")
assert DATA is not None, (
    "CarDD dataset'i bulunamadi (herhangi bir derinlikte 'cardd' iceren "
    "klasor yok). Input paneline bak, dataset gercekten ekli mi kontrol et."
)
print("CarDD kok bulundu:", DATA.relative_to(INPUT))

CARDD = WORK / "data/raw/cardd"
CARDD.mkdir(parents=True, exist_ok=True)

coco_src = next(DATA.rglob("CarDD_COCO"))
if not (CARDD / "CarDD_COCO").exists():
    os.symlink(coco_src, CARDD / "CarDD_COCO")

SOD = CARDD / "CarDD_SOD"
for split in ["TR", "VAL", "TE"]:
    dst_dir = SOD / f"CarDD-{split}"
    dst = dst_dir / f"CarDD-{split}-Mask"
    if dst.exists():
        continue
    hits = list(DATA.rglob(f"CarDD-{split}-Mask"))
    if not hits:
        print(f"  UYARI: CarDD-{split}-Mask bulunamadi")
        continue
    dst_dir.mkdir(parents=True, exist_ok=True)
    os.symlink(hits[0], dst)

for p in ["CarDD_COCO/train2017", "CarDD_COCO/val2017", "CarDD_COCO/test2017",
          "CarDD_SOD/CarDD-TR/CarDD-TR-Mask", "CarDD_SOD/CarDD-VAL/CarDD-VAL-Mask",
          "CarDD_SOD/CarDD-TE/CarDD-TE-Mask"]:
    d = CARDD / p
    n = len(list(d.iterdir())) if d.exists() else "YOK"
    print(f"  {p:<42} {n}")


In [ ]:
# W2'de uretilen S / M1 / M2 (+ yerel M3 = classic) katmanlarini bagla.
#
# UCUNCU DUZELTME -- ASIL KOK NEDEN
# -----------------------------------
# Onceki denemede "synthetic: 0 png" ve "manipulated: 0 png" cikti, ama
# "classic: 520 png" dogru gorundu -- TESADUFEN. Sebep: bu Kaggle Python
# ortaminda `Path.rglob()` SEMBOLIK BAGLI KLASORLERIN ICINE INMIYOR.
#
# synthetic/manipulated icin bagladigimiz sey ALT KLASORLERDI (sd15,
# sd_turbo, sdxl, inpaint_add, inpaint_remove) -- goruntuler bir kat daha
# derinde, rglob oraya inemedigi icin 0 sayildi.
#
# classic icin bagladigimiz sey DOGRUDAN DOSYALARDI (bir kat derine
# inmeye gerek yoktu) -- o yuzden 520 dogru cikti GIBI GORUNDU. Ama
# classic'in kendi ICINDEKI "masks/" alt klasoru de sembolik baglandi --
# O da rglob'a gorunmuyor. Yani maskeler de su an KAYIP, sadece
# fark edilmemisti.
#
# KALICI COZUM: klasorleri HICBIR ZAMAN sembolik baglama. Kaynak agaci
# ozyinelemeli GERCEK dizinlerle yeniden kur, yalnizca YAPRAK DOSYALARI
# sembolik bagla. Boylece rglob/glob/os.walk -- hangisi kullanilirsa
# kullanilsin -- hicbir sembolik-klasore-inme sorunuyla karsilasmaz.

import os
import zipfile


def find_zip(pattern: str):
    return next(INPUT.rglob(pattern), None)


def find_dir(name: str):
    return next((p for p in INPUT.rglob(name) if p.is_dir()), None)


def mirror_files(src_dir, dst_dir):
    """src_dir agacini dst_dir altinda GERCEK klasorlerle yeniden kurar,
    yalnizca dosyalari sembolik baglar. Klasor hicbir zaman sembolik
    olmaz -- rglob'un neden bazen 0 saydigini burada cozuyoruz."""
    dst_dir.mkdir(parents=True, exist_ok=True)
    for item in src_dir.iterdir():
        if item.is_dir():
            mirror_files(item, dst_dir / item.name)
        else:
            link = dst_dir / item.name
            if link.is_symlink() and not link.exists():
                link.unlink()  # kirik/eski bag -- temizle
            if not link.exists():
                os.symlink(item, link)


targets = {
    "data/raw/synthetic": ["w2_synthetic"],
    "data/raw/manipulated": ["w2_manipulated"],
    "data/raw/manipulated/classic": ["w2_classic", "classic"],
}

found_any = False
for rel, aliases in targets.items():
    dst = WORK / rel
    resolved = False

    for base in aliases:
        zpath = find_zip(f"{base}.zip")
        if zpath is not None:
            found_any = resolved = True
            dst.mkdir(parents=True, exist_ok=True)
            if not any(dst.iterdir()):
                with zipfile.ZipFile(zpath) as z:
                    z.extractall(dst)
            break

        dpath = find_dir(base)
        if dpath is not None:
            found_any = resolved = True
            mirror_files(dpath, dst)
            break

    n_png = sum(1 for _ in dst.rglob("*.png")) if dst.exists() else 0
    status = "hazir" if resolved else "BULUNAMADI"
    print(f"  {rel}: {status} ({n_png} png)")

assert found_any, (
    "Hicbir W2 katmani (zip ya da klasor) bulunamadi. Input paneline bak."
)

# M3 (klasik manipulasyon) yerelde uretilmisti ve W2 zip'lerine GIRMEMISTI.
# Burada 0 ise manifest 520 satir eksik cikar ve sha256 TUTMAZ.
m3 = WORK / "data/raw/manipulated/classic"
n_m3 = len(list(m3.rglob("*.png"))) if m3.exists() else 0
n_m3_masks = len(list((m3 / "masks").rglob("*.png"))) if (m3 / "masks").exists() else 0
print(f"\nM3 (classic): {n_m3} png  |  masks/: {n_m3_masks} png")
if n_m3 == 0:
    print("  >>> M3 YOK. Input'a w2_classic/classic dataset'i eklendiginden emin ol.")
if n_m3 > 0 and n_m3_masks == 0:
    print("  >>> UYARI: gorunutuler var ama maskeler yok -- masks/ hala bulunamadi.")


## 2. Manifest + laundering

Bu iki adim yerelde de calistirildi. Burada **ayni ciktiyi** uretmeleri
gerekiyor -- split'ler id hash'inden deterministik turetiliyor.

In [ ]:
!python scripts/build_manifest_v2.py

In [ ]:
!python scripts/apply_laundering.py

### 3b. BUTUNLUK KAPISI — test seti sha256

Yereldeki dondurulmus test setiyle **birebir ayni** manifest uretildi mi?
Tutmuyorsa embedding'ler yerel manifest'e hizalanmaz; **devam etme.**

In [ ]:
# YEREL_SHA -- own_photos HARIC hesaplandi, bilerek.
#
# NEDEN ORIJINAL DONDURULMUS TEST SETININ HASH'I DEGIL
# -------------------------------------------------------
# Yereldeki test_manifest_frozen.sha256, KVKK geregi gizli tutulan 52
# kisisel telefon fotografini (own_photos) da iceriyordu -- bunlar hicbir
# zaman Kaggle'a yuklenmedi (yuklenmemesi dogru, gizlilik nedeniyle).
# Kaggle'in kendi urettigi manifestte bu 52 satir hicbir zaman olusamaz;
# dolayisiyla ORIJINAL hash Kaggle tarafinda YAPISAL OLARAK tutamazdi --
# hangi kod dogru calisirsa calissin.
#
# DOGRULAMA (bu oturumda yapildi): own_photos cikarilinca split sayilari
# Kaggle'in kendi urettigi sayilarla BIREBIR eslesti (train 3520, val 1007,
# test 643). Bu, split atamasinin own_photos'un varligindan/yoklugundan
# ETKILENMEDIGINI kanitlar (hash tabanli, deterministik, own_photos hicbir
# manipulasyona donor degil) -- yani Kaggle'in test seti, yerelin
# own_photos'suz alt kumesiyle TAM ESLESMESI gereken, gecerli bir kontrol.
#
# own_photos'un CLIP embedding'leri SONRADAN yerelde (CPU'da, ~170 gorsel,
# saniyeler surer) ayrica cikarilip clip_probe.py --cache ile Kaggle
# cache'ine EKLENECEK -- atlanmiyor, sadece bu bütünlük kapisinin disinda.

YEREL_SHA = "554ac0bee642d4995272efdbd656cedb1ecc71d044443efc52209b16e9f3d7d3"

!python scripts/build_manifest_v2.py --freeze-test 2>&1 | tail -6

from pathlib import Path
sha_path = Path("data/processed/test_manifest_frozen.sha256")
kaggle_sha = sha_path.read_text().strip() if sha_path.exists() else "(yok)"

print("\nyerel  (own_photos haric):", YEREL_SHA)
print("kaggle                   :", kaggle_sha)
if kaggle_sha == YEREL_SHA:
    print("\n>>> TUTUYOR. Devam edebilirsin.")
else:
    print("\n>>> TUTMUYOR! Manifest farkli uretilmis.")
    print(">>> Muhtemel sebep: M3 (classic) katmani burada yok,")
    print(">>> ya da bir katmanin dosya sayisi farkli. DEVAM ETME,")
    print(">>> once sebebi bul.")


## 4. CLIP embedding cikarma

Dondurulmus CLIP ViT-L/14, 768-d. Parcalar halinde yazilir: oturum koparsa
(W2'de oldugu gibi) kalinan yerden devam eder.

In [ ]:
import subprocess

Path("logs").mkdir(exist_ok=True)

CMD = (
    "python -m src.features.clip_embed "
    "--manifest data/processed/manifest_v2_laundered.parquet "
    "--out data/processed/clip_cache "
    "--batch 64 --device cuda"
)

PROC = globals().get("PROC")
if PROC is not None and PROC.poll() is None:
    print(f"Zaten calisiyor (pid={PROC.pid}), tekrar baslatilmadi.")
else:
    log = open("logs/clip.log", "a")
    PROC = subprocess.Popen(CMD.split(), stdout=log, stderr=subprocess.STDOUT, cwd=str(WORK))
    print(f"Baslatildi (pid={PROC.pid})")
    print("Ilerlemeyi asagidaki hucreyle izle. T4'te ~20-30 dk bekleniyor.")

### 4b. Ilerleme izleme

In [ ]:
import time
from pathlib import Path

def durum():
    print("=" * 68)
    log = Path("logs/clip.log")
    if log.exists():
        satirlar = [s for s in log.read_text(errors="ignore").splitlines() if s.strip()]
        for s in satirlar[-6:]:
            print("  " + s[:108])
    cache = Path("data/processed/clip_cache")
    shards = sorted(cache.glob("shard_*.npy")) if cache.exists() else []
    print(f"\n  yazilan parca : {len(shards)}")
    if shards:
        import numpy as np
        toplam = sum(np.load(s, mmap_mode="r").shape[0] for s in shards)
        print(f"  embedding     : {toplam} / 16796  ({100*toplam/16796:.1f}%)")
    print()
    !nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader

durum()

## 5. Ciktilari indirilebilir hale getir

In [ ]:
import shutil, numpy as np
from pathlib import Path

cache = Path("data/processed/clip_cache")
shards = sorted(cache.glob("shard_*.npy"))
toplam = sum(np.load(s, mmap_mode="r").shape[0] for s in shards)
print(f"{len(shards)} parca, {toplam} embedding")

if toplam < 17176:
    print(f"\n!!! EKSIK: {17176 - toplam} embedding daha bekleniyor.")
    print("!!! Cikarma bitmeden zip alma -- once 4b ile bekle.")
else:
    hedef = "/kaggle/working/w3_clip_cache"
    shutil.make_archive(hedef, "zip", cache)
    mb = Path(hedef + ".zip").stat().st_size / 1e6
    print(f"\n{hedef}.zip  ({mb:.0f} MB)")
    print("\nSonraki adim:")
    print("  1. Save Version -> Quick Save -> 'Save output for this version'")
    print("  2. Output'tan w3_clip_cache.zip indir")
    print("  3. YERELDE: data/processed/clip_cache/ altina ac")
    print("  4. python -m src.detectors.clip_probe --task A")
    print("     python -m src.detectors.clip_probe --task B")